# Deployment — Malware

Este notebook demonstra a fase de **deployment e inferência** do modelo de Malware.

O objetivo é reutilizar o modelo XGBoost já treinado e guardado, sem voltar a executar o treino.

Fluxo apresentado neste notebook:

1. configurar o ambiente;
2. carregar o modelo e as features;
3. testar inferência com amostras reais;
4. testar entradas inválidas;
5. utilizar o agente especializado de Malware;
6. utilizar o agente generativo de relatório;
7. testar o orquestrador;
8. validar o fluxo completo de ponta a ponta.


## 1. Preparação do ambiente


In [ ]:
%pip install -q pandas
%pip install -q xgboost
%pip install -q transformers
%pip install -q torch
%pip install -q accelerate


In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("IAAC-Projeto").exists():
        !git clone https://github.com/diogocarvalho2025056-crypto/IAAC-Projeto.git

    pasta_projeto = Path("IAAC-Projeto").resolve()

else:
    pasta_projeto = Path.cwd().parent.resolve()


if str(pasta_projeto) not in sys.path:
    sys.path.insert(
        0,
        str(pasta_projeto)
    )


print(
    "Pasta do projeto:",
    pasta_projeto
)


## 2. Importar os módulos do projeto

Neste notebook não é implementada novamente a lógica de inferência.

São reutilizados os módulos Python já desenvolvidos no projeto:

- `src/malware_inference.py`
- `src/malware_agent.py`
- `src/report_agent.py`
- `src/orchestrator.py`

Isto segue o requisito de reutilização de funções e módulos Python nos notebooks.


In [ ]:
import pandas as pd

from src.malware_inference import (
    carregar_features,
    carregar_modelo,
    validar_amostra,
    prever_malware
)

from src.malware_agent import (
    analisar_malware
)

from src.report_agent import (
    carregar_gerador,
    gerar_relatorio
)

from src.orchestrator import (
    processar_pedido
)


## 3. Verificar os ficheiros persistidos


In [ ]:
caminho_modelo = (
    pasta_projeto
    / "models"
    / "malware_xgboost.json"
)

caminho_features = (
    pasta_projeto
    / "models"
    / "malware_features.json"
)

print(
    "Modelo existe:",
    caminho_modelo.exists()
)

print(
    "Features existem:",
    caminho_features.exists()
)

print(
    "Modelo:",
    caminho_modelo
)

print(
    "Features:",
    caminho_features
)


O modelo foi guardado em formato JSON nativo do XGBoost. A lista de features foi guardada separadamente para garantir que novas amostras são fornecidas com a estrutura e ordem esperadas pelo classificador.


## 4. Carregar o modelo e as features


In [ ]:
features = carregar_features()
modelo = carregar_modelo()

print(
    "Número de features:",
    len(features)
)

print(
    "\nFeatures esperadas:"
)

for feature in features:
    print(
        "-",
        feature
    )

print(
    "\nModelo carregado:",
    type(modelo).__name__
)


## 5. Preparar amostras reais para teste


In [ ]:
ficheiro_csv = (
    pasta_projeto
    / "dataset"
    / "Malware_prepared.csv"
)

df = pd.read_csv(
    ficheiro_csv
)


amostra_benigna = (
    df[df["Malicious"] == 0]
    .iloc[0]
    .drop("Malicious")
    .to_dict()
)


amostra_malware = (
    df[df["Malicious"] == 1]
    .iloc[0]
    .drop("Malicious")
    .to_dict()
)


print(
    "Amostra benigna preparada."
)

print(
    "Amostra malware preparada."
)


## 6. Teste de inferência — amostra benigna


In [ ]:
resultado_benigno = prever_malware(
    amostra_benigna
)

resultado_benigno


Resultado esperado: classificação `Benigno`, com elevada confiança. Nos testes realizados durante o desenvolvimento, a amostra utilizada apresentou aproximadamente **99,95% de confiança** para a classe Benigno.


## 7. Teste de inferência — amostra malware


In [ ]:
resultado_malware = prever_malware(
    amostra_malware
)

resultado_malware


Resultado esperado: classificação `Malware`, com elevada confiança. Nos testes realizados durante o desenvolvimento, a amostra utilizada apresentou aproximadamente **99,88% de confiança** para a classe Malware.


## 8. Testes de robustez

Antes da previsão, a pipeline valida a entrada.

São testados os seguintes casos:

- feature em falta;
- valor não numérico;
- feature inesperada;
- valor NaN;
- entrada que não é um dicionário.


### 8.1 Feature em falta


In [ ]:
amostra_sem_feature = amostra_malware.copy()

amostra_sem_feature.pop(
    "ImageBase"
)

prever_malware(
    amostra_sem_feature
)


### 8.2 Valor não numérico


In [ ]:
amostra_valor_invalido = amostra_malware.copy()

amostra_valor_invalido[
    "ImageBase"
] = "texto_invalido"

prever_malware(
    amostra_valor_invalido
)


### 8.3 Feature inesperada


In [ ]:
amostra_feature_extra = amostra_malware.copy()

amostra_feature_extra[
    "FeatureInventada"
] = 123

prever_malware(
    amostra_feature_extra
)


### 8.4 Valor NaN


In [ ]:
amostra_nan = amostra_malware.copy()

amostra_nan[
    "ImageBase"
] = float("nan")

prever_malware(
    amostra_nan
)


### 8.5 Entrada inválida


In [ ]:
prever_malware(
    [
        "isto",
        "não",
        "é",
        "um",
        "dicionário"
    ]
)


Os erros são devolvidos de forma controlada. O objetivo é impedir que dados inválidos cheguem ao XGBoost e provoquem exceções durante a previsão.


## 9. Agente especializado de Malware


In [ ]:
agente_benigno = analisar_malware(
    amostra_benigna
)

agente_malware = analisar_malware(
    amostra_malware
)


print(
    "Resultado benigno:"
)

print(
    agente_benigno
)


print(
    "\nResultado malware:"
)

print(
    agente_malware
)


O agente especializado não substitui o classificador.

A responsabilidade mantém-se separada:

- **XGBoost** → decide a classe;
- **malware_agent.py** → estrutura e apresenta o resultado.


## 10. Agente generativo de relatório


O agente generativo utiliza **Transformers / Hugging Face** e o modelo **Qwen2.5-1.5B-Instruct**.

O LLM não decide se a amostra é Malware ou Benigno. Recebe apenas a decisão já produzida pelo XGBoost e transforma-a num pequeno relatório em linguagem natural.


In [ ]:
print(
    "A carregar modelo generativo..."
)

gerador = carregar_gerador()

print(
    "Modelo generativo carregado."
)


In [ ]:
relatorio = gerar_relatorio(
    agente_malware,
    gerador
)

print(
    relatorio
)


O prompt do agente impõe regras para limitar alucinações:

- não alterar a classificação;
- não inventar features;
- não inventar métricas;
- não confundir confiança de uma previsão com Accuracy global;
- escrever em português de Portugal;
- indicar que a previsão não constitui certeza absoluta.


## 11. Testar o orquestrador


In [ ]:
resultado_orquestrador = processar_pedido(
    "malware",
    amostra_malware
)

resultado_orquestrador


O orquestrador coordena o fluxo entre os vários componentes.

Fluxo atual:

`pedido → orchestrator.py → malware_agent.py → malware_inference.py → XGBoost → report_agent.py → relatório`


## 12. Apresentar o resultado de forma simples


In [ ]:
resultado_tecnico = resultado_orquestrador[
    "resultado"
]

print(
    "Classificação:",
    resultado_tecnico[
        "classificacao"
    ]
)

print(
    "Confiança:",
    f"{resultado_tecnico['confianca'] * 100:.2f}%"
)

print(
    "Probabilidade Benigno:",
    f"{resultado_tecnico['probabilidade_benigno'] * 100:.2f}%"
)

print(
    "Probabilidade Malware:",
    f"{resultado_tecnico['probabilidade_malware'] * 100:.2f}%"
)

print(
    "\nRelatório:"
)

print(
    resultado_orquestrador[
        "relatorio"
    ]
)


## 13. Estado atual

A pipeline de Malware encontra-se funcional desde o modelo persistido até à geração do relatório:

1. modelo XGBoost guardado;
2. features guardadas;
3. inferência sem novo treino;
4. validação de entradas;
5. testes de robustez;
6. agente especializado de Malware;
7. agente generativo;
8. orquestrador;
9. interface `main.py`.

Os agentes de Ransomware e Phishing serão acrescentados ao mesmo orquestrador quando os respetivos modelos estiverem concluídos.


## 14. Integração futura

Após a integração dos três classificadores, será criada uma versão do sistema multiagente utilizando uma framework adicional, como:

- OpenClaw;
- CrewAI;
- n8n.

A mesma arquitetura poderá posteriormente ser ligada a uma interface externa, por exemplo Telegram.

O objetivo é manter a separação entre:

- classificadores de Machine Learning;
- agentes especializados;
- agente generativo;
- orquestração;
- interface com o utilizador.
